# 권현성의 멀티턴·멀티쿼리·도구 선택 실습

- **핵심 개념:** 대화 맥락 복원, 복합 질문 분해, 도구 라우팅
- **나의 수정:** 모호한 실습 프롬프트를 구체화하고 `eval`을 안전한 `json.loads`로 교체
- API 키는 코드에 저장하지 않고 Colab `Secrets`의 `OPENAI_API_KEY`를 사용합니다.


## 1. 멀티턴을 반영한 쿼리 생성기

이전 질문, 이전 답변, 현재 질문을 바탕으로 멀티턴 대화를 생성하는 쿼리 생성기를 작성하세요.  

답변으로 검색어만 반환하면 됩니다.

In [ ]:
from openai import OpenAI
import json

import os
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Colab 왼쪽의 Secrets에 OPENAI_API_KEY를 등록하세요.")

os.environ["OPENAI_API_KEY"] = api_key


In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
def generate_search_query(previous_question: str, previous_answer: str, current_question: str) -> str:
    """
    GPT-4를 이용해 멀티턴 대화 기반 검색어를 재구성하는 함수
    :param previous_question: 이전 질문
    :param previous_answer: 이전 답변
    :param current_question: 현재 질문
    :return: 검색어 리스트
    """
    prompt = f"""
    이전 질문: {previous_question}
    이전 답변: {previous_answer}
    현재 질문: {current_question}

    현재 질문의 생략된 대상을 이전 대화로 보완해, 검색에 바로 사용할 한 문장으로 다시 작성하세요.
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "당신은 대화 맥락을 보존하는 검색어 재작성기입니다. 설명 없이 검색어만 반환하세요."},
            {"role": "user", "content": prompt}
        ],
    )

    return response.choices[0].message.content.strip()

In [ ]:
# 사용 예시
previous_question = "서울 여행지 추천좀"
previous_answer = "네 추천해드릴게요"
current_question = "전주는?"

search_query = generate_search_query(previous_question, previous_answer, current_question)
print(search_query)

In [ ]:
# 테스트용 샘플 데이터 (튜플 형태)
sample_data = [
    ("서울 여행지 추천좀", "네 추천해드릴게요", "전주는?"),
    ("부산 맛집 알려줘", "부산에서 유명한 맛집은~", "그럼 제주도는?"),
    ("한옥마을 추천", "전주 한옥마을이 유명합니다", "서울에도 있어?"),
    ("강원도 여행 어디가 좋아?", "설악산과 속초가 좋습니다", "그럼 바다도 추천해줘"),
    ("제주 카페 추천해줘", "오설록 티하우스가 인기 많아요", "한라산 근처는?"),
    ("겨울 여행지 추천", "강원도의 평창이 좋아요", "그럼 따뜻한 곳은?"),
    ("여름 휴가 어디가 좋아?", "제주도가 좋아요", "동해는 어때?"),
    ("서울 명소 알려줘", "경복궁과 남산타워가 있습니다", "야경 좋은 곳은?"),
    ("가족 여행지 추천", "에버랜드가 괜찮아요", "놀이공원 말고 자연도 있나?"),
    ("국내 드라이브 코스 추천", "춘천 가는 길이 좋습니다", "제주에서도 알려줘")
]

# 샘플 데이터 반복 호출
for idx, (prev_q, prev_a, curr_q) in enumerate(sample_data, 1):
    result = generate_search_query(prev_q, prev_a, curr_q)
    print(f"테스트 {idx}: {result}")

## 2. 멀티 쿼리 생성기

주어진 입력으로부터 다수의 질문으로 분리되도록 해보세요.  

['검색어1', '검색어2', '검색어3']으로 분리되어야 합니다.

In [ ]:
def generate_multi_queries(current_query: str) -> list[str]:
    """복합 질문을 독립적으로 검색 가능한 JSON 검색어 배열로 분리합니다."""
    prompt = f"""
    다음 문장을 주제별 독립 검색어로 분리하세요.
    사용자 문장: {current_query}
    출력은 반드시 문자열 JSON 배열만 사용하세요. 예: ["검색어 1", "검색어 2"]
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "복합 질문을 빠짐없이 나누는 검색 쿼리 설계자입니다."},
            {"role": "user", "content": prompt},
        ],
    )
    return json.loads(response.choices[0].message.content.strip())


In [ ]:
current_query = "서울과 부산의 2박 3일 여행 코스를 각각 알려줘"
search_queries = generate_multi_queries(current_query)


In [ ]:
search_queries

In [ ]:
search_queries_json = """["서울 인기 맛집 추천", "부산 현지인 맛집 리스트"]"""
print(json.loads(search_queries_json))


In [ ]:
# 멀티 쿼리 테스트 데이터
multi_query_samples = [
    "서울 여행지랑 전주 여행지랑 제주 여행지 알려줘",
    "부산 해수욕장이랑 강원도 계곡 추천해줘",
    "봄에 가기 좋은 여행지랑 가을 단풍 명소 알려줘",
    "유럽 여행지랑 미국 관광 명소 알려줘",
    "서울 맛집이랑 부산 맛집이랑 대구 맛집 추천해줘",
    "겨울 캠핑 장소랑 여름 캠핑 장소 알려줘",
    "아이랑 가기 좋은 곳이랑 부모님 모시고 가기 좋은 곳 알려줘",
    "국내 힐링 여행지랑 해외 힐링 여행지 알려줘",
    "혼자 여행하기 좋은 곳이랑 친구랑 가기 좋은 곳 추천해줘",
    "제주도 드라이브 코스랑 부산 드라이브 코스 알려줘"
]

# 멀티 쿼리 샘플 반복 호출
for idx, query in enumerate(multi_query_samples, 1):
    queries = generate_multi_queries(query)
    print(f"멀티 쿼리 테스트 {idx}: {queries}")

## 3. 멀티턴과 멀티쿼리 대응

여러분들은 멀티턴과 멀티 쿼리를 모두 처리 가능한 질문 생성기를 만들 겁니다.  

검색어(번호): 완성된 검색어

형태로 검색어가 나오도록 프롬프트를 만드세요.

In [ ]:
def generate_multi_turn_multi_queries(previous_query: str, previous_answer: str, current_query: str) -> list:
    """
    GPT-4를 이용해 멀티턴 & 멀티 쿼리를 생성하는 함수
    :param previous_query: 이전 입력
    :param previous_answer: 이전 응답
    :param current_query: 현재 입력
    :return: 검색어 리스트
    """
    prompt = f"""
    이전 대화의 지시 대상을 복원하고, 현재 입력에 포함된 요청을 독립 검색어로 나누세요.
    이전 입력: {previous_query}
    이전 답변: {previous_answer}
    현재 입력: {current_query}

    이전 대화의 지시 대상을 복원하고, 현재 입력에 포함된 요청을 독립 검색어로 나누세요.
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "대화 맥락을 반영해 여러 검색어를 만드는 쿼리 설계자입니다. JSON 배열만 반환하세요."},
            {"role": "user", "content": prompt}
        ],
    )

    search_queries = response.choices[0].message.content.strip()
    return json.loads(search_queries)

# 사용 예시
previous_query = "서울 여행지 추천"
previous_answer = "서울 여행지를 추천해드릴게요~"
current_query = "전주도 알려주고 제주 관광 상품도 알려주고 서울 맛집도 알려줘"

search_queries = generate_multi_turn_multi_queries(previous_query, previous_answer, current_query)

for idx, query in enumerate(search_queries, 1):
    print(f"검색어{idx}: {query}")

In [ ]:
multi_turn_multi_query_samples = [
    ("서울 여행지 추천해줘", "경복궁과 남산이 좋습니다.",
     "그럼 거기서 야경 보기 좋은 곳은 어디고, 거기 아니면 전주나 제주에서 가볼 만한 곳은?"),

    ("부산 맛집 알려줘", "해운대와 서면 쪽이 유명합니다.",
     "그럼 거기서 분위기 좋은 카페는 어디고, 거기 아니더라도 광안리 근처 맛집 추천해줘."),

    ("강원도 해수욕장 추천", "양양과 속초 쪽이 좋습니다.",
     "그럼 거기 근처에서 캠핑 가능한 곳은 어디고, 거기 아니더라도 아이들이 놀기 좋은 곳은?"),

    ("국내 겨울 여행지 추천", "평창과 강릉이 좋습니다.",
     "그럼 거기서 온천 가능한 곳은 어디고, 거기 아니면 남쪽 지방의 따뜻한 여행지는?"),

    ("서울 근교 드라이브 코스", "양평과 남양주가 좋습니다.",
     "그럼 거기 근처 맛집은 어디고, 아니면 제주나 부산에서 드라이브하기 좋은 곳은?"),

    ("제주 관광 명소 추천", "성산일출봉과 오설록이 좋습니다.",
     "그럼 거기 근처에서 가족이 가기 좋은 곳은 어디고, 아니면 동쪽 지역 명소는?"),

    ("가족 여행지 추천", "에버랜드와 롯데월드가 좋습니다.",
     "그럼 거기서 가까운 자연 속 숙소는 어디고, 아니면 강원도나 충청도 지역 추천해줘."),

    ("봄꽃 명소 추천", "경주와 진해가 좋습니다.",
     "그럼 거기서 사진 찍기 좋은 곳은 어디고, 거기 아니더라도 가을 단풍 명소는 어디가 좋을까?"),

    ("역사 여행지 추천", "부여와 경주가 좋습니다.",
     "그럼 거기서 아이와 같이 갈만한 곳은 어디고, 거기 아니더라도 서울 근처 역사 명소는?"),

    ("나 서울 놀러갈거야. 그러니까 서울 맛집 추천해줘", "홍대가 좋습니다.",
     "오 젊음의 거리지. 거기서 분위기 좋은 식당은 어디고, 아니면 거기 아니라도 되니까 디저트가 맛있는 카페는?")
]

# 반복 호출 예시
for idx, (prev_q, prev_a, curr_q) in enumerate(multi_turn_multi_query_samples, 1):
    queries = generate_multi_turn_multi_queries(prev_q, prev_a, curr_q)
    print(f"멀티턴 & 멀티쿼리 테스트 {idx}: {queries}")

## 4. 도구 매칭

여러분들은 이전 질문, 이전 답변, 현재 질문으로부터 도구를 매칭할 겁니다.  

도구 결과는 "[도구 이름]: [완성된 검색어]" 형태로 나와야 합니다.

In [ ]:
def match_queries_to_tools(tools, previous_query, previous_answer, current_query):
    """
    GPT-4를 이용해 쿼리를 분석하고, 도구들과 매칭하여 검색어를 생성하는 함수
    :param tools: 도구 목록 [{"name": str, "description": str}, ...]
    :param previous_query: 이전 질문
    :param previous_answer: 이전 답변
    :param current_query: 현재 질문
    :return: {도구명: 검색어} 형태의 리스트
    """
    tool_info = "\n".join(
        [f"- {tool['name']}: {tool['description']}" for tool in tools]
    )

    prompt = f"""
    대화 맥락을 참고해 각 요청을 가장 적합한 도구와 검색어에 연결하세요.

    사용 가능한 도구 목록은 다음과 같아:
    {tool_info}

    이전 질문: "{previous_query}"
    이전 답변: "{previous_answer}"
    현재 질문: "{current_query}"

    대화 맥락을 참고해 각 요청을 가장 적합한 도구와 검색어에 연결하세요.
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "요청을 분해하고 도구별 검색어를 설계하세요. 도구명과 검색어만 간결하게 반환하세요."},
            {"role": "user", "content": prompt}
        ],
    )

    matched_results = response.choices[0].message.content.strip().split("\n")
    return [line.strip() for line in matched_results if line.strip()]

In [ ]:
# 도구 정의
tools = [
    {"name": "TourSpotFinder", "description": "관광지나 산책로 등 여행 및 휴식 장소를 추천합니다."},
    {"name": "FoodieFinder", "description": "맛집이나 카페 등 음식과 관련된 장소를 추천합니다."},
    {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요."},
]

# 예시 입력
previous_query = "서울 관광지 추천해줘"
previous_answer = "홍대 어때요?"
current_query = "그럼 근처에 산책하기 좋은 곳이랑 분위기 좋은 식당은 어디고, 거기 아니더라도 디저트가 맛있는 카페는?"

# 함수 호출
result = match_queries_to_tools(tools, previous_query, previous_answer, current_query)

# 결과 출력
for item in result:
    print(item)

In [ ]:
# 도구 정의 (각 도메인에 맞게 예시마다 다름)
tool_examples = [
    # 1. 여행
    [
        {"name": "TourSpotFinder", "description": "관광지에 대한 정보를 조회하거나, 관광지나 산책로 등 여행지를 추천합니다."},
        {"name": "FoodieFinder", "description": "맛집이나 카페를 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 2. 영화 및 문화
    [
        {"name": "MovieRecommender", "description": "영화나 드라마를 추천합니다."},
        {"name": "CultureEventFinder", "description": "문화 행사 및 전시회를 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 3. 쇼핑
    [
        {"name": "FashionShopFinder", "description": "옷과 액세서리 매장을 추천합니다."},
        {"name": "ElectronicsFinder", "description": "전자제품 쇼핑몰을 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 4. 스포츠
    [
        {"name": "SportEventFinder", "description": "스포츠 경기 일정을 추천합니다."},
        {"name": "FitnessCenterFinder", "description": "운동 시설을 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 5. 교육 및 강좌
    [
        {"name": "OnlineCourseFinder", "description": "온라인 강좌 및 강의를 추천합니다."},
        {"name": "LocalWorkshopFinder", "description": "지역 워크숍과 세미나를 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 6. 부동산
    [
        {"name": "HouseFinder", "description": "부동산 매물 및 집을 추천합니다."},
        {"name": "InteriorDesignFinder", "description": "인테리어 관련 업체를 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 7. 금융
    [
        {"name": "InvestmentAdvisor", "description": "투자 상품과 관련 정보를 제공합니다."},
        {"name": "BankFinder", "description": "은행 및 금융기관 정보를 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 8. 의료 및 건강
    [
        {"name": "HospitalFinder", "description": "병원 및 진료 시설을 추천합니다."},
        {"name": "PharmacyFinder", "description": "약국을 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 9. IT 및 기술
    [
        {"name": "TechSupportFinder", "description": "기술 지원 및 수리 업체를 추천합니다."},
        {"name": "SoftwareFinder", "description": "소프트웨어 제품을 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 10. 취미 및 여가
    [
        {"name": "HobbyClassFinder", "description": "취미 클래스와 강좌를 추천합니다."},
        {"name": "LeisureActivityFinder", "description": "여가 활동 및 체험을 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ]
]

In [ ]:
# 예시 데이터 (멀티턴 & 멀티쿼리)
test_samples = [
    ("서울 여행 추천해줘", "북촌 한옥마을이 좋습니다.",
     "거기서 근처 카페랑 산책로 추천해줘, 아니면 전주 한옥마을은? 그리고 너 누구야?"),

    ("최근 재밌는 영화 추천", "오펜하이머가 흥행했습니다.",
     "거기 감독 다른 영화랑 근처 문화 행사도 알려줘 그리고 크리스토퍼 놀란에 대한 논란도 궁금하네"),

    ("쇼핑할 옷가게 추천해줘", "강남에 편집샵이 많아요.",
     "거기서 가까운 전자제품 매장도 알려줘, 아니면 홍대 쪽도? 그리고 갤럭시 S25 최신 기능은?"),

    ("이번 주 스포츠 경기 추천해줘", "축구 K리그 경기가 많습니다.",
     "거기 근처 피트니스 센터는 어디고, 야구 일정도 알려줘. 야구 선수 중에 누가 야구 경기 제일 잘하냐?"),

    ("온라인에서 배울 만한 강좌 추천", "코세라와 인프런이 좋습니다.",
     "지역에서 참여 가능한 세미나는 뭐가 있고, 영어 강좌는 어디서 들을 수 있어? 그리고 영어 1타 강사 누구냐?"),

    ("서울에서 살기 좋은 아파트 추천해줘", "강남구와 마포구가 인기입니다.",
     "인테리어 업체는 어디가 좋고, 집 가격 시세는 어디서 확인할까? 그리고 요즘 집값 어떠냐?"),

    ("투자할 만한 금융상품 추천", "ETF나 채권이 안전합니다.",
     "좋아. 그럼 그런거 은행 상담 가능한 곳은 어디고, 요즘 금리 정보는 어디서 볼까? 그리고 엔비디아 창업자 누구냐?"),

    ("가까운 병원 추천해줘", "서울대병원이 가깝습니다.",
     "거기 근처 약국은 어디고, 건강검진 가능한 병원은? 그리고 강남에서 가장 유명한 카페 어디냐?"),

    ("컴퓨터가 고장났는데 어디 갈까?", "용산에 수리 업체가 많습니다.",
     "거기서 추천할 만한 소프트웨어는 뭐가 있고, 인터넷 문제는 어디서 해결하지? 인터넷 기사 추천 해줄래?"),

    ("서울에 취미 활동할 곳 추천해줘", "홍대에 클래스가 많습니다.",
     "거기서 체험 가능한 여가 활동은 뭐가 있고, 다른 지역은? 그리고 동호회 추천해볼래?")
]

# 반복 호출로 테스트
for idx, ((prev_q, prev_a, curr_q), tools) in enumerate(zip(test_samples, tool_examples), 1):
    print(f"\n--- 테스트 {idx} ---")
    matched_queries = match_queries_to_tools(tools, prev_q, prev_a, curr_q)
    for mq in matched_queries:
        print(mq)